In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from tqdm import tqdm
from sklearn.feature_selection import SelectKBest, f_regression
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load cleaned + engineered data
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks"

df_train = pd.read_parquet(f"{DATA_DIR}/train_fe.parquet")
df_val   = pd.read_parquet(f"{DATA_DIR}/val_fe.parquet")
df_test  = pd.read_parquet(f"{DATA_DIR}/test_fe.parquet")

def sample_searches(df, n_rows=50_000, random_state=42):
    """Subsample complete search groups instead of random rows, so NDCG@5 still
    evaluates real rankings.
    """
    rng = np.random.default_rng(random_state)
    # Randomly order search IDs; each srch_id contains all properties shown for one search.
    srch_ids = df["srch_id"].drop_duplicates().to_numpy()
    rng.shuffle(srch_ids)

    # Count how many rows/properties belong to each search.
    sizes = df.groupby("srch_id").size()
    selected_ids = []
    total_rows = 0

    # Keep adding full searches until the subsample has about n_rows observations.
    for srch_id in srch_ids:
        selected_ids.append(srch_id)
        total_rows += sizes.loc[srch_id]
        if total_rows >= n_rows:
            break

    # Return all rows for the selected searches, preserving complete ranking groups.
    return df[df["srch_id"].isin(selected_ids)].reset_index(drop=True)

df_train = sample_searches(df_train)
df_val = sample_searches(df_val)
df_test = sample_searches(df_test)

print(f"Number of rows in df_train: {len(df_train)}")
df_train.head()

Number of rows in df_train: 50021


,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,ump,total_fee,per_fee,score2ma,score1d2,count_window,prop_id_1cnt,srch_destination_id_1cnt,prop_location_score2_1cnt,relevance
0,631,2013-03-11 09:54:54,5,106,0.0,0.0,73,4178,3,4.0,...,29.026855,115.0,115.0,-1.175444,0.012000,1028,161,9732,392738,0
1,631,2013-03-11 09:54:54,5,106,0.0,0.0,73,5117,3,4.5,...,-18.241425,111.0,111.0,-1.183621,0.013842,1028,28,9732,392738,0
2,631,2013-03-11 09:54:54,5,106,0.0,0.0,73,6016,4,4.5,...,-27.825638,187.0,187.0,-1.525010,0.016600,1028,32,9732,2582738,0
3,631,2013-03-11 09:54:54,5,106,0.0,0.0,73,10311,2,3.5,...,37.731613,85.0,85.0,-0.098124,0.001856,1028,76,9732,494019,0
4,631,2013-03-11 09:54:54,5,106,0.0,0.0,73,12929,4,4.5,...,27.389893,135.0,135.0,-1.152957,0.011625,1028,230,9732,392738,0


In [ ]:
# Define features, X, and y
target_col = "relevance"
exclude_cols = ["click_bool", "booking_bool", "gross_bookings_usd", "position", "relevance"]
id_cols = ["srch_id", "date_time", "prop_id"]
feature_cols = [c for c in df_train.columns if c not in exclude_cols + id_cols]

X_train, y_train = df_train[feature_cols], df_train["relevance"]
X_val, y_val = df_val[feature_cols], df_val["relevance"]
X_test = df_test[feature_cols]

In [ ]:
# Feature selection
selector = SelectKBest(score_func=f_regression, k=20)
selector.fit(X_train.fillna(0), y_train)

selected_features = np.array(feature_cols)[selector.get_support()]
print(f"Selected features:\n{selected_features}")

X_train = X_train[selected_features]
X_val   = X_val[selected_features]
X_test  = X_test[selected_features]

Selected features:
['prop_starrating' 'prop_review_score' 'prop_location_score2'
 'promotion_flag' 'srch_length_of_stay' 'random_bool' 'price_usd_clipped'
 'log_price' 'price_per_person' 'price_rank_in_query'
 'price_rel_to_query_mean' 'star_rank_in_query' 'review_rank_in_query'
 'loc1_rank_in_query' 'loc2_rank_in_query' 'ump' 'per_fee' 'score2ma'
 'score1d2' 'prop_location_score2_1cnt']


In [ ]:
# Define helper function for NDCG@5 score
def get_cdg(scores):
    return sum(r / np.log2(i + 2) for i, r in enumerate(scores))


def get_ndcg(df, cutoff=5):
    scores = []

    for _, group in df.groupby("srch_id"):
        group = group.sort_values("rank")
        actual = group["relevance"].values[:cutoff]
        ideal = np.sort(group["relevance"].values)[::-1][:cutoff]

        idcg = get_cdg(ideal)
        if idcg == 0:
            continue

        scores.append(get_cdg(actual) / idcg)

    return np.mean(scores) if scores else 0.0

In [ ]:
# Tune k = # groups
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

best_k, best_ndcg = None, -np.inf

for k in tqdm([5, 10, 25, 50, 100]):
    knn = KNeighborsRegressor( # Justify parameter choices
        n_neighbors = k,
        weights = 'distance',
        metric = 'euclidean',
        algorithm = 'ball_tree',
        n_jobs = -1
    )

    knn.fit(X_train_scaled, y_train)

    df_val_copy = df_val.copy()
    df_val_copy["pred_relevance"] = knn.predict(X_val_scaled)
    df_val_copy["rank"] = df_val_copy.groupby("srch_id")["pred_relevance"].rank(ascending = False, method = "first")

    ndcg = get_ndcg(df_val_copy, cutoff=5)

    print(f"Best k = {best_k},   Best NDCG@5 = {best_ndcg:.3f}")

    if ndcg > best_ndcg:
        best_ndcg = ndcg
        best_k = k

print(f"Best k = {best_k},   Best NDCG@5 = {best_ndcg:.3f}")


 20%|██        | 1/5 [00:47<03:11, 47.97s/it]

Best k = None,   Best NDCG@5 = -inf


 40%|████      | 2/5 [01:36<02:25, 48.50s/it]

Best k = 5,   Best NDCG@5 = 0.233


 60%|██████    | 3/5 [02:46<01:55, 57.98s/it]

Best k = 10,   Best NDCG@5 = 0.252


 80%|████████  | 4/5 [03:55<01:02, 62.64s/it]

Best k = 25,   Best NDCG@5 = 0.269


100%|██████████| 5/5 [05:06<00:00, 61.32s/it]

Best k = 50,   Best NDCG@5 = 0.283
Best k = 100,   Best NDCG@5 = 0.306


In [ ]:
# Fit final model w/ best k
df = pd.concat([df_train, df_val], ignore_index = True)
X = df[selected_features]
y = df["relevance"]

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor( # Keeping same params as before
        n_neighbors = best_k,
        weights = 'distance',
        metric = 'euclidean',
        algorithm = 'ball_tree',
        n_jobs = -1
    ))
])

knn_pipeline.fit(X, y)

Pipeline(steps=[('scaler', StandardScaler()),
                ('knn',
                 KNeighborsRegressor(algorithm='ball_tree', metric='euclidean',
                                     n_jobs=-1, n_neighbors=100,
                                     weights='distance'))])

In [ ]:
# Generate submission
df_test_copy = df_test.copy()
df_test_copy["pred_relevance"] = knn_pipeline.predict(X_test)

submission = (
    df_test_copy
    .sort_values(["srch_id", "pred_relevance"], ascending = [True, False])[["srch_id", "prop_id"]]
    .rename(columns={"srch_id": "SearchId", "prop_id": "PropertyId"})
)

submission.to_csv(DATA_DIR + "/submission_knn.csv", index = False)